# Actividad Formativa (Opcional) · Semana 9
## Limpieza de un dataset con pandas

```yaml
# CONFIG
FULL_NAME: Isabella Ramos Betancourt
GITHUB_USER: Isarb-21
```

**Isabella Ramos Betancourt** · Facultad de Ingeniería, CORHUILA  
Ciencia de Datos (Cód. 69109) [Pénsum 40D, Grupo 1] · Profesor: Jesús Ariel Gonzáles Bonilla · Octubre de 2026

**Objetivos:** detectar y corregir problemas de calidad en un dataset de telemetría IoT, documentar el antes/después y comentar 2 dimensiones de calidad que se mejoraron.

**Uso:** ejecutar con *Restart & Run All* en la carpeta donde está `lectura_sensor_raw.csv`. Todas las cifras salen de la ejecución.

### 1. Carga del dataset y diagnóstico (ANTES)

In [1]:
import pandas as pd
import numpy as np

raw = pd.read_csv("lectura_sensor_raw.csv")

print("=== ESTADO ANTES ===")
print("Número de filas:", len(raw))
print("Filas duplicadas:", raw.duplicated().sum())
print("\nNulos por columna (isnull):")
print(raw.isnull().sum())
print("\nTipos de datos:")
print(raw.dtypes)
print("\nVariantes de id_lote:", sorted(raw["id_lote"].unique()))

=== ESTADO ANTES ===
Número de filas: 75
Filas duplicadas: 3

Nulos por columna (isnull):
id_lectura           0
id_lote              0
fecha_hora           0
humedad_suelo_pct    2
temperatura_c        1
dtype: int64

Tipos de datos:
id_lectura             int64
id_lote                  str
fecha_hora               str
humedad_suelo_pct    float64
temperatura_c            str
dtype: object

Variantes de id_lote: ['LOT 001', 'LOT-001', 'LOT-002', 'LOT-003', 'LOT001', 'lot002', 'lote-003']


In [2]:
# Problemas que isnull() no ve: texto en una columna numérica y valores imposibles
temp_num = pd.to_numeric(raw["temperatura_c"], errors="coerce")
hum = raw["humedad_suelo_pct"]

antes = {
    "filas": len(raw),
    "duplicados exactos": int(raw.duplicated().sum()),
    "variantes de id_lote": int(raw["id_lote"].nunique()),
    "nulos en humedad": int(hum.isna().sum()),
    "nulos en temperatura (vacíos)": int(raw["temperatura_c"].isna().sum()),
    "temperatura con texto inválido": int((temp_num.isna() & raw["temperatura_c"].notna()).sum()),
    "humedad fuera de [0, 100]": int(((hum > 100) | (hum < 0)).sum()),
    "temperatura fuera de [0, 50]": int((temp_num.notna() & ((temp_num > 50) | (temp_num < 0))).sum()),
    "tipo de fecha_hora": str(raw["fecha_hora"].dtype),
    "tipo de temperatura_c": str(raw["temperatura_c"].dtype),
}
formatos = raw["fecha_hora"].astype(str).str.replace(r"\d", "9", regex=True).value_counts()
print("Formatos de fecha_hora encontrados:")
print(formatos.to_string())

Formatos de fecha_hora encontrados:
fecha_hora
9999-99-99 99:99    72
99/99/9999 99:99     1
9999/99/99 99:99     1
99-99-9999 99:99     1


### 2. Proceso de limpieza

Reglas: `id_lote` canónico `LOT-XXX`; `fecha_hora` a `datetime` (las fechas ISO se leen aparte de las de día/mes/año); `temperatura_c` numérica; humedad válida en [0, 100] %; temperatura válida en [0, 50] °C; faltantes imputados con la **mediana por lote**.

In [3]:
df = raw.copy()

# 1. Normalizar id_lote: extraer el número y reconstruir el código canónico
num = df["id_lote"].astype(str).str.extract(r"(\d+)")[0]
assert num.notna().all(), "Hay id_lote sin número"
df["id_lote"] = num.astype(int).map("LOT-{:03d}".format)

# 2. fecha_hora: ISO (AAAA-MM-DD o AAAA/MM/DD) por un lado, día/mes/año por otro.
#    Con format="mixed" a secas, 01/09/2026 se leería como 9 de enero.
txt = df["fecha_hora"].astype(str).str.strip()
es_iso = txt.str.match(r"^\d{4}[-/]\d{2}[-/]\d{2}")
df["fecha_hora"] = pd.to_datetime(txt.where(es_iso).str.replace("/", "-"), format="ISO8601").fillna(
    pd.to_datetime(txt.where(~es_iso), format="mixed", dayfirst=True))
print("Rango de fechas:", df["fecha_hora"].min(), "->", df["fecha_hora"].max())
assert df["fecha_hora"].dt.strftime("%Y-%m").nunique() == 1, "Fechas en meses distintos"

# 3. temperatura_c a numérico (el texto inválido pasa a NaN)
df["temperatura_c"] = pd.to_numeric(df["temperatura_c"], errors="coerce")

# 4. Quitar duplicados (ya con claves y fechas normalizadas)
n0 = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print("Duplicados eliminados:", n0 - len(df))

# 5. Valores fuera de rango -> NaN
m_hum = (df["humedad_suelo_pct"] > 100) | (df["humedad_suelo_pct"] < 0)
m_tmp = df["temperatura_c"].notna() & ((df["temperatura_c"] > 50) | (df["temperatura_c"] < 0))
print("Fuera de rango -> humedad:", int(m_hum.sum()), "| temperatura:", int(m_tmp.sum()))
df.loc[m_hum, "humedad_suelo_pct"] = np.nan
df.loc[m_tmp, "temperatura_c"] = np.nan

# 6. Imputar con la mediana por lote
imputados = int(df[["humedad_suelo_pct", "temperatura_c"]].isna().sum().sum())
print("Valores imputados:", imputados)
for col in ["humedad_suelo_pct", "temperatura_c"]:
    df[col] = df.groupby("id_lote")[col].transform(lambda s: s.fillna(s.median()))

# 7. Guardar el dataset limpio
df.to_csv("lectura_sensor_clean.csv", index=False)

Rango de fechas: 2026-09-01 00:00:00 -> 2026-09-01 23:00:00
Duplicados eliminados: 3
Fuera de rango -> humedad: 2 | temperatura: 0
Valores imputados: 7


### 3. Reporte ANTES vs. DESPUÉS

In [4]:
print("=== ESTADO DESPUÉS ===")
print("Número de filas:", len(df))
print("Filas duplicadas:", df.duplicated().sum())
print("\nNulos por columna:")
print(df.isnull().sum())
print("\nTipos de datos:")
print(df.dtypes)
print("\nLotes:", sorted(df["id_lote"].unique()))
print("Rango humedad_suelo_pct:", df["humedad_suelo_pct"].min(), "-", df["humedad_suelo_pct"].max())
print("Rango temperatura_c    :", df["temperatura_c"].min(), "-", df["temperatura_c"].max())

assert df.isnull().sum().sum() == 0 and df.duplicated().sum() == 0
assert df["humedad_suelo_pct"].between(0, 100).all() and df["temperatura_c"].between(0, 50).all()

=== ESTADO DESPUÉS ===
Número de filas: 72
Filas duplicadas: 0

Nulos por columna:
id_lectura           0
id_lote              0
fecha_hora           0
humedad_suelo_pct    0
temperatura_c        0
dtype: int64

Tipos de datos:
id_lectura                    int64
id_lote                         str
fecha_hora           datetime64[us]
humedad_suelo_pct           float64
temperatura_c               float64
dtype: object

Lotes: ['LOT-001', 'LOT-002', 'LOT-003']
Rango humedad_suelo_pct: 28.1 - 55.8
Rango temperatura_c    : 19.1 - 27.6


In [5]:
despues = {
    "filas": len(df),
    "duplicados exactos": int(df.duplicated().sum()),
    "variantes de id_lote": int(df["id_lote"].nunique()),
    "nulos en humedad": int(df["humedad_suelo_pct"].isna().sum()),
    "nulos en temperatura (vacíos)": int(df["temperatura_c"].isna().sum()),
    "temperatura con texto inválido": 0,
    "humedad fuera de [0, 100]": int(((df["humedad_suelo_pct"] > 100) | (df["humedad_suelo_pct"] < 0)).sum()),
    "temperatura fuera de [0, 50]": int(((df["temperatura_c"] > 50) | (df["temperatura_c"] < 0)).sum()),
    "tipo de fecha_hora": str(df["fecha_hora"].dtype),
    "tipo de temperatura_c": str(df["temperatura_c"].dtype),
}
print("Valores imputados en total:", imputados)
pd.DataFrame({"ANTES": antes, "DESPUÉS": despues})

Valores imputados en total: 7


,ANTES,DESPUÉS
filas,75,72
duplicados exactos,3,0
variantes de id_lote,7,3
nulos en humedad,2,0
nulos en temperatura (vacíos),1,0
temperatura con texto inválido,2,0
"humedad fuera de [0, 100]",2,0
"temperatura fuera de [0, 50]",0,0
tipo de fecha_hora,str,datetime64[us]
tipo de temperatura_c,str,float64


### 4. Dimensiones de calidad mejoradas

1. **Exactitud (accuracy):** 2 lecturas de humedad eran físicamente imposibles (150.0 % y 135.5 %); se marcaron como nulas y se reemplazaron por la mediana de su lote. Eso las vuelve **plausibles**, pero son **estimaciones**, no mediciones. También se corrigieron las fechas escritas como día/mes/año: leídas como mes/día habrían quedado en enero en lugar de septiembre, y la verificación del rango de fechas lo comprueba.
2. **Consistencia (consistency):** `id_lote` tenía 7 variantes escritas distintas (`LOT-001`, `LOT 001`, `LOT001`, `lot002`, `lote-003`, entre otras) para solo 3 parcelas; ahora hay 3 códigos canónicos (`LOT-001`, `LOT-002`, `LOT-003`), y por eso los agrupamientos por lote y las imputaciones por lote son correctos. Las fechas también pasaron de varios formatos a un único `datetime64`.

*Otras dimensiones tocadas:* **completitud** (7 valores faltantes o inválidos imputados, 0 nulos al final), **unicidad** (3 duplicados exactos eliminados) y **validez** (`temperatura_c` pasó de texto a número).